In [ ]:
# Multi-Assignment — Security Observation Allocation
# --------------------------------------------------
# Each service must be monitored by multiple agents

from optilab.aliases import Model, GRB, quicksum
import numpy as np

# -------------------------------------------------------------------------------
# Model
# -------------------------------------------------------------------------------
m = Model()

# -------------------------------------------------------------------------------
# Problem Data (Enterprise Monitoring System)
# -------------------------------------------------------------------------------
rng = np.random.default_rng(47)

n_services = rng.integers(20, 40)
n_agents = rng.integers(6, 12)

# monitoring cost (agent i monitoring service j)
cost = rng.integers(1, 10, size=(n_agents, n_services))

# -------------------------------------------------------------------------------
# Redundancy requirements (THIS is new vs vertex cover)
# -------------------------------------------------------------------------------
# each service must be monitored by at least l[j] agents
l = rng.integers(1, 3, size=n_services)

# each service can have at most u[j] monitors (optional upper bound)
u = l + rng.integers(1, 3, size=n_services)

# each agent has limited capacity (workload constraint)
capacity = rng.integers(4, 10, size=n_agents)

# -------------------------------------------------------------------------------
# Decision Variables
# -------------------------------------------------------------------------------
# x[i][j] = agent i monitors service j
x = [
    [m.add_binary_var(name=f"x_{i}_{j}") for j in range(n_services)]
    for i in range(n_agents)
]

# -------------------------------------------------------------------------------
# Objective (minimize monitoring cost)
# -------------------------------------------------------------------------------
m.set_objective(
    quicksum(
        cost[i][j] * x[i][j]
        for i in range(n_agents)
        for j in range(n_services)
    ),
    GRB.MINIMIZE
)

# -------------------------------------------------------------------------------
# Constraints
# -------------------------------------------------------------------------------

# (1) Service-side redundancy constraints (KEY difference from vertex cover)
for j in range(n_services):
    m.add_constraint(
        quicksum(x[i][j] for i in range(n_agents)) >= l[j]
    )
    m.add_constraint(
        quicksum(x[i][j] for i in range(n_agents)) <= u[j]
    )

# (2) Agent workload constraints
for i in range(n_agents):
    m.add_constraint(
        quicksum(x[i][j] for j in range(n_services)) <= capacity[i]
    )

# -------------------------------------------------------------------------------
# Solve
# -------------------------------------------------------------------------------
m.solve()

# -------------------------------------------------------------------------------
# Extract solution
# -------------------------------------------------------------------------------
solution = np.array([[x[i][j].x for j in range(n_services)]
                     for i in range(n_agents)])

# compute coverage statistics
service_coverage = solution.sum(axis=0)

print(f"Backend:        {m.backend_name}")
print(f"Services:       {n_services}")
print(f"Agents:         {n_agents}")

print("\nService coverage (min requirement vs achieved):")
for j in range(min(10, n_services)):
    print(f"  Service {j}: {service_coverage[j]} / {l[j]} required")

print("\nTotal assignments:", int(solution.sum()))

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# -------------------------------------------------------------------
# Build bipartite graph
# -------------------------------------------------------------------
B = nx.Graph()

agents = list(range(n_agents))
services = list(range(n_services))

B.add_nodes_from(agents, bipartite=0)
B.add_nodes_from([n_agents + j for j in services], bipartite=1)

# Add assignment edges
for i in agents:
    for j in services:
        if solution[i][j] > 0.5:
            B.add_edge(i, n_agents + j)

# -------------------------------------------------------------------
# Layout (clean bipartite separation)
# -------------------------------------------------------------------
pos = {}

# Left side: agents
pos.update((i, (0, i - len(agents) / 2)) for i in agents)

# Right side: services
pos.update(
    (n_agents + j, (24, j - len(services) / 2))
    for j in services
)

# -------------------------------------------------------------------
# Plot
# -------------------------------------------------------------------
plt.figure(figsize=(18, 12))
ax = plt.gca()

# IMPORTANT: prevents matplotlib from collapsing layout
ax.set_aspect('equal', adjustable='box')

# Nodes
nx.draw_networkx_nodes(
    B, pos,
    nodelist=agents,
    node_color="skyblue",
    node_size=600,
    label="Agents"
)

nx.draw_networkx_nodes(
    B, pos,
    nodelist=[n_agents + j for j in services],
    node_color="lightgreen",
    node_size=600,
    label="Services"
)

nx.draw_networkx_edges(
    B, pos,
    alpha=0.35,
    width=1.2,
    connectionstyle="arc3,rad=0.08",
    arrows=True
)

# Labels
labels = {i: f"A{i}" for i in agents}
labels.update({n_agents + j: f"S{j}" for j in services})

nx.draw_networkx_labels(B, pos, labels=labels, font_size=9)

# -------------------------------------------------------------------
# Hard axis control (CRITICAL)
# -------------------------------------------------------------------
plt.xlim(-1, 26)
plt.ylim(
    -max(len(agents), len(services)) / 2 - 1,
    max(len(agents), len(services)) / 2 + 1
)

plt.axis("off")
plt.title("Multi-Assignment: Monitoring Allocation Graph")
plt.show()